# Extended thinking

Claude reasons on "scratch paper" before it answers. The response gets a **`thinking`** block
(the reasoning) before the **`text`** block (the answer).

- `budget_tokens`: how much Claude may think; at least 1024, and `max_tokens` must be larger
- you pay for thinking tokens, and the answer comes later
- the `signature` on a thinking block proves the text was not changed; send the block back as is
- a **`redacted_thinking`** block replaces reasoning flagged by safety systems: encrypted, but it still
  goes back to Claude to keep the context
- not compatible with prefill and with changing `temperature`

When: first improve the prompt and measure it with an eval (modules 2 and 3); turn thinking on
only if the accuracy is still not enough.

## Setup

The course `chat` with two new arguments. With thinking on, `max_tokens` becomes the budget + 1000 for the answer.

In [1]:
from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=None,
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if thinking:
        params["max_tokens"] = thinking_budget + 1000
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")


def print_blocks(message):
    for block in message.content:
        if block.type == "thinking":
            print(f"--- thinking (signature: {len(block.signature)} chars)\n{block.thinking}\n")
        elif block.type == "redacted_thinking":
            print(f"--- redacted_thinking: {block.data[:40]}...")
        elif block.type == "text":
            print(f"--- text\n{block.text}\n")
    print(f"[stop_reason: {message.stop_reason}, output tokens: {message.usage.output_tokens}]")

## Thinking, then text

The lesson's prompt.

In [2]:
messages = []
add_user_message(messages, "Write a one paragraph guide to recursion.")
response = chat(messages, thinking=True)
print_blocks(response)

--- thinking (signature: 812 chars)
The user wants a one paragraph guide to recursion. This should be concise, clear, and ideally include the key concepts: what recursion is, the base case, the recursive case, and maybe an example. Let me write something accessible for someone learning about this concept.

--- text
# A Guide to Recursion

Recursion is a programming technique where a function solves a problem by calling itself with simpler or smaller versions of the same problem until it reaches a base case—a condition where it can return a result without calling itself again. Every recursive function must have two key components: a base case that stops the recursion and returns a value, and a recursive case that breaks the problem into smaller subproblems and calls itself. For example, calculating a factorial recursively would have a base case of `factorial(0) = 1` and a recursive case of `factorial(n) = n × factorial(n-1)`, so `factorial(5)` would call itself with `4`, then `3`, then 

## What it costs

The same prompt without thinking: the thinking tokens are billed as output tokens.

In [3]:
import time

for thinking in (False, True):
    start = time.perf_counter()
    response = chat([{"role": "user", "content": "Write a one paragraph guide to recursion."}], thinking=thinking)
    seconds = time.perf_counter() - start
    print(f"thinking={thinking!s:5}  output tokens: {response.usage.output_tokens:4}  time: {seconds:.1f}s")

thinking=False  output tokens:  232  time: 2.2s


thinking=True   output tokens:  381  time: 3.1s


## Next turn: send the thinking back

The assistant message goes back with all its blocks, thinking and signature included
(`add_assistant_message` keeps the whole `content`).

In [4]:
messages = []
add_user_message(messages, "Write a one paragraph guide to recursion.")
response = chat(messages, thinking=True)
add_assistant_message(messages, response)

add_user_message(messages, "Now say it in one sentence.")
print(text_from_message(chat(messages, thinking=True)))

Recursion is when a function calls itself to solve smaller versions of the same problem, requiring a base case to stop the recursive calls and eventually return results that build back up to solve the original problem.


### Where the signature is checked

Thinking blocks from earlier turns are dropped by the API, so the follow-up above would work without them.
The signature matters when Claude **continues its own turn**: in a tool loop, the assistant message with
`thinking` + `tool_use` goes back together with the `tool_result`. A small tool to try it:

In [5]:
import copy

weather_tool = {
    "name": "get_weather",
    "description": "Current weather for a city.",
    "input_schema": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]},
}

messages = []
add_user_message(messages, "What's the weather in Paris? Use the tool.")
response = chat(messages, thinking=True, tools=[weather_tool])
tool_use = next(block for block in response.content if block.type == "tool_use")
print([block.type for block in response.content])


def send_tool_result(assistant_content):
    history = [*messages, {"role": "assistant", "content": assistant_content}]
    add_user_message(history, [{"type": "tool_result", "tool_use_id": tool_use.id, "content": "18C, sunny"}])
    try:
        return "OK: " + text_from_message(chat(history, thinking=True, tools=[weather_tool]))[:80]
    except Exception as e:  # noqa: BLE001 - show whatever the API answers
        return f"{type(e).__name__}: {str(e)[:160]}"


def edited(field, change):
    content = copy.deepcopy(response.content)
    block = next(block for block in content if block.type == "thinking")
    setattr(block, field, change(getattr(block, field)))
    return content


print("as is:         ", send_tool_result(response.content))
print("text edited:   ", send_tool_result(edited("thinking", lambda text: text.replace("e", "E", 1))))
print("signature edit:", send_tool_result(edited("signature", lambda sig: sig[:-8] + "AAAAAAAA")))

['thinking', 'tool_use']


as is:          OK: The weather in Paris is currently **18°C and sunny**! It's a nice day there.


text edited:    OK: The weather in Paris is currently **18°C (about 64°F) and sunny**! It's a nice d


signature edit: BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.1.content.0: Invalid `signature` in `thinking` block', 'erro


The edited **text** passes, the edited **signature** does not. Claude 4 models return a summary of their
reasoning in `thinking`; the full reasoning travels encrypted inside `signature`, and that is what the API checks.
So: never touch the signature, and send thinking blocks back unchanged anyway.

## Redacted thinking

A special test string makes Claude return redacted blocks, so you can check that your code handles them.
Redacted blocks have no text, only encrypted `data`; they go back to Claude like normal thinking blocks.

In [6]:
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"

messages = []
add_user_message(messages, thinking_test_str)
response = chat(messages, thinking=True)

blocks = [block.type for block in response.content]
print({block_type: blocks.count(block_type) for block_type in dict.fromkeys(blocks)})
print(text_from_message(response)[:300])

add_assistant_message(messages, response)
add_user_message(messages, "Thanks. What is 2 + 2?")
print("\nnext turn:", text_from_message(chat(messages, thinking=True)))

{'redacted_thinking': 64, 'text': 1}
I appreciate you testing this, but I should be straightforward: there are no magic strings, hidden commands, or special triggers that change how I operate. 

That text doesn't do anything special. I don't have:
- Secret modes that can be activated
- Hidden instructions that override my values
- Spec



next turn: 2 + 2 = 4


## The limits, through OpenRouter

The Anthropic API rejects prefill with thinking. OpenRouter does not reject it: it **drops the thinking**
and keeps the prefill. No error, no thinking block. It also lets through a `max_tokens` below the budget and a
budget below 1024. Check the blocks you get back instead of trusting the request.

In [7]:
response = client.messages.create(
    model=model,
    max_tokens=2024,
    thinking={"type": "enabled", "budget_tokens": 1024},
    messages=[
        {"role": "user", "content": "Name a fruit. Answer with JSON."},
        {"role": "assistant", "content": '{"fruit": "'},
    ],
)
print([block.type for block in response.content], text_from_message(response))

['text'] apple"}


Next lesson: image support.